# 🛡️ 165 AI 智能防詐大模型 - LoRA / QLoRA 領域微調實戰

**專題名稱**：165 AI 智能多模態防詐騙鑑識系統 (Anti-Scam LLM System)  
**核心任務**：將台灣 165 反詐騙官方知識庫與攻防案例，透過 **QLoRA (4-bit LoRA)** 深度注入開源 8B 大語言模型（Llama-3-Taiwan / Qwen2.5），訓練出專屬的台灣在地化防詐鑑識大腦！

### 🌟 本實戰 Notebook 亮點：
1. **極速微調**：使用 Unsloth / QLoRA 技術，在 Google Colab 免費 T4 GPU (16GB) 僅需 **20~30 分鐘** 即可完成微調！
2. **格式 100% 穩定**：模型學會原生結構化輸出，無需長 System Prompt 即可秒出標準 JSON 鑑識報告。
3. **一鍵導出 GGUF / Ollama**：支援將模型權重打包為 GGUF 格式，可部署於邊緣設備或本機離線運行。

## 📌 步驟 1：安裝極速微調套件環境 (Unsloth & Hugging Face)

In [ ]:
%%capture
# 安裝 Unsloth、PyTorch 與 Hugging Face 微調生態系
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 📌 步驟 2：載入開源基底模型 (Base LLM) 與 4-bit 量化

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # 支援長文本鑑識
dtype = None # 自動判斷 (Tesla T4 為 Float16, Ampere 以上為 Bfloat16)
load_in_4bit = True # 啟用 4-bit QLoRA 節省 70% 顯存

# 選擇基底模型 (支援 Llama-3-8B 或 Qwen2.5-7B)
model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✅ 4-bit 基底模型載入成功！")

## 📌 步驟 3：設定 LoRA Adapter 參數 (注入 165 防詐專屬權重)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA Rank (16 可兼顧表現與訓練速度)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, # Unsloth 優化設定 0 最佳
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("✅ LoRA 權重配置完成！")

## 📌 步驟 4：準備 165 防詐 SFT 微調資料集 (Dataset Formatting)

In [ ]:
from datasets import Dataset
import json

# 提示詞模板
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# 載入由專案產生的訓練語料 (可直接上傳 data/train_alpaca.json)
sample_data = [
    {
        "instruction": "你是由中華民國警政署 165 反詐騙諮詢專線標準所訓練的「165 AI 智能防詐鑑識專家」。請精確剖析輸入內容並輸出嚴謹的 JSON 結構鑑識報告。",
        "input": "【台北地檢署通知】台端涉嫌洗錢防制法案，請於本日 17:00 前攜帶存摺至地檢署說明，或點此 http://tp-prosecutor-gov.top 進行線上資產公證監管，逾期將拘提。",
        "output": json.dumps({
            "risk_score": 98,
            "primary_scam_type": "假冒公務機關 / 檢警監管帳戶",
            "red_flags": ["非政府 .gov.tw 網址 (使用 .top 可疑域名)", "恐嚇核發拘票並限期壓迫", "要求線上資產公證/監管帳戶"],
            "evidence_analysis": "檢警司法機關絕不會透過簡訊傳送公文或要求線上資產公證監管。",
            "actionable_guidance": "1. 立即掛斷勿點擊連結 2. 撥打 165 專線求證。"
        }, ensure_ascii=False, indent=2)
    },
    {
        "instruction": "你是由中華民國警政署 165 反詐騙諮詢專線標準所訓練的「165 AI 智能防詐鑑識專家」。請精確剖析輸入內容並輸出嚴謹的 JSON 結構鑑識報告。",
        "input": "【台股實戰學院】張老師親自帶盤！佈局下半年翻倍黑馬飆股，上週帶會員獲利達 45%。免費領取精準進出場點位，名額僅限前 20 名：https://line.me/ti/p/scam888",
        "output": json.dumps({
            "risk_score": 95,
            "primary_scam_type": "假投資飆股 / 虛擬幣",
            "red_flags": ["保證高額獲利 45%", "假冒投顧老師名義", "引導加入私人 LINE 群組"],
            "evidence_analysis": "典型飆股詐騙，以名師帶盤與限額製造飢餓感，誘導至未受監管平台。",
            "actionable_guidance": "合法投顧絕不透過私人 LINE 群報明牌，切勿匯款。"
        }, ensure_ascii=False, indent=2)
    },
    {
        "instruction": "你是由中華民國警政署 165 反詐騙諮詢專線標準所訓練的「165 AI 智能防詐鑑識專家」。請精確剖析輸入內容並輸出嚴謹的 JSON 結構鑑識報告。",
        "input": "【國泰世華銀行】您於 2026/09/07 20:15 在 PChome 24h 購物刷卡消費 NT$ 1,890 元。如非本人交易請立即致電本行客服 (02)2383-1000。",
        "output": json.dumps({
            "risk_score": 5,
            "primary_scam_type": "非詐騙 / 正常銀行刷卡通知",
            "red_flags": [],
            "evidence_analysis": "具備精確消費資訊與官方客服電話，無釣魚網址。",
            "actionable_guidance": "正常消費通知，無需採取動作。"
        }, ensure_ascii=False, indent=2)
    }
]

dataset = Dataset.from_list(sample_data)
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"✅ 資料集格式化完成，範例筆數：{len(dataset)}")

## 📌 步驟 5：啟動 SFT 訓練 (Training Execution)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 開始微調訓練...")
trainer_stats = trainer.train()
print("🎉 訓練完成！")

## 📌 步驟 6：實測微調後 165 AI 防詐大腦 (Live Inference Test)

In [ ]:
FastLanguageModel.for_inference(model)

test_input = "【健保局通知】您的健保卡存在異常違規重複領藥，即將於 24 小時內鎖卡！請點擊 http://nhi-tw-verify.cc 進行身分驗證。"

inputs = tokenizer(
[
    alpaca_prompt.format(
        "你是由中華民國警政署 165 反詐騙諮詢專線標準所訓練的「165 AI 智能防詐鑑識專家」。請精確剖析輸入內容並輸出嚴謹的 JSON 結構鑑識報告。",
        test_input,
        "", # 留空由模型生成
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
result = tokenizer.batch_decode(outputs)
print("==================== 165 AI 鑑識報告 ====================")
print(result[0].split("### Response:")[1].replace("<|eot_id|>", "").strip())

## 📌 步驟 7：儲存 LoRA 權重或匯出 GGUF 供本地離線部署

In [ ]:
# 1. 儲存 LoRA Adapter
model.save_pretrained("165_lora_model")
tokenizer.save_pretrained("165_lora_model")
print("✅ LoRA Adapter 已儲存至 165_lora_model 資料夾！")

# 2. (可選) 一鍵匯出為 16-bit 完整合併模型或 GGUF 格式供 Ollama 使用
# model.save_pretrained_merged("165_anti_scam_full", tokenizer, save_method = "merged_16bit")
# model.save_pretrained_gguf("165_anti_scam_gguf", tokenizer, quantization_method = "q4_k_m")